# HCC miRNA biomarker analysis: reproduction notebook

This notebook reproduces the core analysis of

> Jabri A, …, **Mujtaba T**, …, Yaqinuddin A. *Integrative analysis of tissue and circulating miRNAs as biomarkers for progression and survival in hepatocellular carcinoma.* **Non-coding RNA Research** 16, 167–177 (2026). [doi:10.1016/j.ncrna.2025.11.002](https://doi.org/10.1016/j.ncrna.2025.11.002)

using only public data:

| Data | Source | Used for |
|---|---|---|
| TCGA-LIHC miRNA-seq (tumour + adjacent normal) and clinical data | [GDC API](https://api.gdc.cancer.gov) | Differential expression, survival, stage/metastasis subgroups |
| Serum miRNA microarray, GSE113740 | [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE113740) | Circulating miRNAs, HCC vs non-cancer, AFP and stage |

**Note:** this is an independent Python re-implementation written for this repository, not the original analysis code. Differential expression uses Welch's t-test on log2 expression with Benjamini–Hochberg correction instead of limma-trend, so exact numbers can differ slightly from the paper.

**How to run:** open in Google Colab → *Runtime → Run all*. Data download takes about 5–10 minutes. Tables are written to `results/` and figures to `plots/`.

In [ ]:
!pip install -q lifelines

In [ ]:
import io, json, gzip, tarfile, os, re, time
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy import stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# ---- Settings ----
PROJECT = "TCGA-LIHC"
GEO_ID = "GSE113740"
LOG2FC_CUTOFF = 1.0      # |log2 fold change| for differential expression
FDR_CUTOFF = 0.05        # Benjamini–Hochberg adjusted p-value
MIN_RPM = 1.0            # a miRNA must reach this RPM ...
MIN_FRACTION = 0.5       # ... in at least this fraction of samples to be analysed
PAPER_SERUM_MIRNAS = ["hsa-miR-3619-3p", "hsa-miR-1290", "hsa-miR-1185-2-3p"]

os.makedirs("results", exist_ok=True)
os.makedirs("plots", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10})
GDC = "https://api.gdc.cancer.gov"


def bh(pvals):
    """Benjamini–Hochberg adjusted p-values (NaN-safe)."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    if ok.any():
        out[ok] = stats.false_discovery_control(p[ok], method="bh")
    return out


def welch_de(expr, group_a, group_b):
    """Welch t-test per row of a log2 expression matrix. Fold change is group_a vs group_b."""
    a, b = expr[group_a], expr[group_b]
    t, p = stats.ttest_ind(a, b, axis=1, equal_var=False, nan_policy="omit")
    res = pd.DataFrame({
        "mean_A": a.mean(axis=1),
        "mean_B": b.mean(axis=1),
        "log2FC": a.mean(axis=1) - b.mean(axis=1),
        "t": np.asarray(t, dtype=float),
        "p_value": np.asarray(p, dtype=float),
    }, index=expr.index)
    res["FDR"] = bh(res["p_value"])
    return res.sort_values("p_value")

## 1. Download TCGA-LIHC miRNA expression from the GDC

In [ ]:
def gdc_query_files():
    filters = {"op": "and", "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": [PROJECT]}},
        {"op": "in", "content": {"field": "data_type", "value": ["miRNA Expression Quantification"]}},
    ]}
    params = {
        "filters": json.dumps(filters),
        "fields": "file_id,file_name,cases.submitter_id,cases.samples.submitter_id,cases.samples.sample_type",
        "format": "JSON", "size": "5000",
    }
    r = requests.get(f"{GDC}/files", params=params, timeout=120)
    r.raise_for_status()
    rows = []
    for h in r.json()["data"]["hits"]:
        case = h["cases"][0]
        sample = case["samples"][0]
        rows.append({"file_id": h["file_id"], "file_name": h["file_name"],
                     "case_id": case["submitter_id"], "sample_id": sample["submitter_id"],
                     "sample_type": sample["sample_type"]})
    return pd.DataFrame(rows)


def gdc_download(file_ids, batch=50):
    """Download quantification files; returns {file_id: DataFrame}."""
    out = {}
    for i in range(0, len(file_ids), batch):
        ids = list(file_ids[i:i + batch])
        for attempt in range(3):
            try:
                r = requests.post(f"{GDC}/data", json={"ids": ids}, timeout=600)
                r.raise_for_status()
                break
            except requests.RequestException:
                if attempt == 2:
                    raise
                time.sleep(5)
        if len(ids) == 1:
            out[ids[0]] = pd.read_csv(io.BytesIO(r.content), sep="\t")
        else:
            with tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz") as tar:
                for m in tar.getmembers():
                    if m.isfile() and m.name.endswith(".txt") and "/" in m.name:
                        fid = m.name.split("/")[0]
                        out[fid] = pd.read_csv(tar.extractfile(m), sep="\t")
        print(f"  downloaded {min(i + batch, len(file_ids))}/{len(file_ids)} files")
    return out


files = gdc_query_files()
files = files[files["sample_type"].isin(["Primary Tumor", "Solid Tissue Normal"])]
print(files["sample_type"].value_counts())

quant = gdc_download(files["file_id"].tolist())
rpm = pd.DataFrame({fid: df.set_index("miRNA_ID")["reads_per_million_miRNA_mapped"]
                    for fid, df in quant.items()})
rpm.columns = files.set_index("file_id").loc[rpm.columns, "sample_id"]
rpm = rpm.T.groupby(level=0).mean().T          # merge any duplicate aliquots per sample

sample_info = files.drop_duplicates("sample_id").set_index("sample_id").loc[rpm.columns]
keep = (rpm >= MIN_RPM).mean(axis=1) >= MIN_FRACTION
log_expr = np.log2(rpm.loc[keep] + 1)
print(f"{log_expr.shape[0]} miRNAs expressed in ≥{MIN_FRACTION:.0%} of {log_expr.shape[1]} samples")

## 2. Differential expression: tumour vs adjacent normal tissue

In [ ]:
tumour = sample_info.index[sample_info["sample_type"] == "Primary Tumor"]
normal = sample_info.index[sample_info["sample_type"] == "Solid Tissue Normal"]
print(f"Tumour: {len(tumour)}  Normal: {len(normal)}")

de = welch_de(log_expr, tumour, normal).rename(columns={"mean_A": "mean_tumour", "mean_B": "mean_normal"})
de["direction"] = np.where(de["log2FC"] > 0, "up", "down")
sig = de[(de["FDR"] < FDR_CUTOFF) & (de["log2FC"].abs() >= LOG2FC_CUTOFF)]
de.to_csv("results/tissue_DE_tumour_vs_normal.csv")
sig.to_csv("results/tissue_DE_significant.csv")
print(f"{len(sig)} significant miRNAs ({(sig['direction'] == 'up').sum()} up, {(sig['direction'] == 'down').sum()} down)")
sig.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
y = -np.log10(de["FDR"].clip(lower=1e-300))
is_sig = de.index.isin(sig.index)
ax.scatter(de["log2FC"][~is_sig], y[~is_sig], s=8, c="#B0B0B0", lw=0)
ax.scatter(de["log2FC"][is_sig], y[is_sig], s=12,
           c=np.where(de["log2FC"][is_sig] > 0, "#C0392B", "#2E6DB4"), lw=0)
for name in sig.head(8).index:
    ax.annotate(name, (de.loc[name, "log2FC"], y[name]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axvline(LOG2FC_CUTOFF, ls="--", c="k", lw=0.6); ax.axvline(-LOG2FC_CUTOFF, ls="--", c="k", lw=0.6)
ax.axhline(-np.log10(FDR_CUTOFF), ls="--", c="k", lw=0.6)
ax.set_xlabel("log2 fold change (tumour vs normal)"); ax.set_ylabel("-log10 FDR")
ax.set_title("TCGA-LIHC tissue miRNAs")
fig.tight_layout(); fig.savefig("plots/tissue_volcano.png"); plt.show()

In [ ]:
if len(sig) >= 2:
    top = sig.index[:30]
    order = list(normal) + list(tumour)
    z = log_expr.loc[top, order]
    z = z.sub(z.mean(axis=1), axis=0).div(z.std(axis=1), axis=0).clip(-3, 3)
    fig, ax = plt.subplots(figsize=(8, 0.25 * len(top) + 1.5))
    im = ax.imshow(z.values, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top, fontsize=7)
    ax.set_xticks([len(normal) / 2, len(normal) + len(tumour) / 2]); ax.set_xticklabels(["Normal", "Tumour"])
    ax.axvline(len(normal) - 0.5, c="k", lw=1)
    fig.colorbar(im, ax=ax, label="z-score", shrink=0.6)
    ax.set_title("Top differentially expressed miRNAs")
    fig.tight_layout(); fig.savefig("plots/tissue_heatmap.png"); plt.show()

## 3. Clinical data and overall survival

In [ ]:
def gdc_query_cases():
    filters = {"op": "in", "content": {"field": "project.project_id", "value": [PROJECT]}}
    params = {
        "filters": json.dumps(filters),
        "fields": ("submitter_id,demographic.vital_status,demographic.days_to_death,"
                   "diagnoses.days_to_last_follow_up,diagnoses.ajcc_pathologic_stage,"
                   "diagnoses.ajcc_pathologic_m,follow_ups.days_to_follow_up"),
        "format": "JSON", "size": "5000",
    }
    r = requests.get(f"{GDC}/cases", params=params, timeout=120)
    r.raise_for_status()
    rows = []
    for h in r.json()["data"]["hits"]:
        demo = h.get("demographic") or {}
        diags = h.get("diagnoses") or []
        fups = h.get("follow_ups") or []
        follow = [d.get("days_to_last_follow_up") for d in diags] + [f.get("days_to_follow_up") for f in fups]
        follow = [v for v in follow if v is not None]
        stage = next((d.get("ajcc_pathologic_stage") for d in diags if d.get("ajcc_pathologic_stage")), None)
        m_stage = next((d.get("ajcc_pathologic_m") for d in diags if d.get("ajcc_pathologic_m")), None)
        rows.append({"case_id": h["submitter_id"], "vital_status": demo.get("vital_status"),
                     "days_to_death": demo.get("days_to_death"),
                     "days_to_last_follow_up": max(follow) if follow else None,
                     "stage": stage, "m_stage": m_stage})
    return pd.DataFrame(rows).set_index("case_id")


clin = gdc_query_cases()
clin["event"] = (clin["vital_status"] == "Dead").astype(int)
clin["time_days"] = np.where(clin["event"] == 1, clin["days_to_death"], clin["days_to_last_follow_up"])
clin["time_days"] = pd.to_numeric(clin["time_days"], errors="coerce")
clin["stage_group"] = clin["stage"].str.extract(r"Stage (IV|I{1,3})", expand=False)
clin["metastasis"] = clin["m_stage"].map(lambda m: "M1" if isinstance(m, str) and m.startswith("M1")
                                         else ("M0" if m == "M0" else np.nan))
clin.to_csv("results/clinical_summary.csv")
print(clin["stage_group"].value_counts(dropna=False).sort_index())

# one primary tumour per patient
tum_info = sample_info.loc[tumour].copy()
tum_info = tum_info[~tum_info["case_id"].duplicated()]
surv = clin.loc[clin.index.intersection(tum_info["case_id"])]
surv = surv[surv["time_days"] > 0].dropna(subset=["time_days"])
case_to_sample = tum_info.reset_index().set_index("case_id")["sample_id"]
print(f"Patients with survival data: {len(surv)} ({surv['event'].sum()} deaths)")

In [ ]:
surv_rows = []
for mir in sig.index:
    x = log_expr.loc[mir, case_to_sample.loc[surv.index]].values
    d = surv[["time_days", "event"]].copy()
    d["expr_z"] = (x - x.mean()) / x.std()
    d["high"] = (x > np.median(x)).astype(int)
    lr = logrank_test(d.loc[d.high == 1, "time_days"], d.loc[d.high == 0, "time_days"],
                      d.loc[d.high == 1, "event"], d.loc[d.high == 0, "event"])
    cph = CoxPHFitter().fit(d[["time_days", "event", "expr_z"]], "time_days", "event")
    s = cph.summary.loc["expr_z"]
    surv_rows.append({"miRNA": mir, "logrank_p": lr.p_value, "HR_per_SD": s["exp(coef)"],
                      "HR_lower95": s["exp(coef) lower 95%"], "HR_upper95": s["exp(coef) upper 95%"],
                      "cox_p": s["p"]})

surv_res = pd.DataFrame(surv_rows).set_index("miRNA").sort_values("cox_p")
surv_res["cox_FDR"] = bh(surv_res["cox_p"])
surv_res["effect"] = np.where(surv_res["HR_per_SD"] > 1, "risk", "protective")
surv_res.to_csv("results/tissue_survival.csv")
print(f"{(surv_res['cox_p'] < 0.05).sum()} of {len(surv_res)} DE miRNAs associated with overall survival (Cox p < 0.05)")
surv_res.head(15)

In [ ]:
top_surv = surv_res.index[:4]
if len(top_surv):
    fig, axes = plt.subplots(1, len(top_surv), figsize=(3.6 * len(top_surv), 3.3), squeeze=False)
    for ax, mir in zip(axes[0], top_surv):
        x = log_expr.loc[mir, case_to_sample.loc[surv.index]].values
        high = x > np.median(x)
        for mask, label, col in [(high, "High", "#C0392B"), (~high, "Low", "#2E6DB4")]:
            KaplanMeierFitter().fit(surv["time_days"][mask] / 365.25, surv["event"][mask],
                                    label=f"{label} (n={mask.sum()})").plot_survival_function(ax=ax, ci_show=False, color=col)
        ax.set_title(f"{mir}\nlog-rank p = {surv_res.loc[mir, 'logrank_p']:.3g}", fontsize=9)
        ax.set_xlabel("Years"); ax.set_ylabel("Overall survival"); ax.set_ylim(0, 1.02)
    fig.tight_layout(); fig.savefig("plots/tissue_km_curves.png"); plt.show()

## 4. Clinical subgroups: tumour stage and metastasis

In [ ]:
tum_cases = case_to_sample.index.intersection(clin.index)
sub = clin.loc[tum_cases, ["stage_group", "metastasis"]]
sub_rows = []
for mir in sig.index:
    vals = log_expr.loc[mir, case_to_sample.loc[tum_cases]].values
    df_ = sub.assign(expr=vals)
    groups = [g["expr"].values for _, g in df_.dropna(subset=["stage_group"]).groupby("stage_group") if len(g) >= 3]
    kw_p = stats.kruskal(*groups).pvalue if len(groups) >= 2 else np.nan
    m1, m0 = df_.loc[df_.metastasis == "M1", "expr"], df_.loc[df_.metastasis == "M0", "expr"]
    mw_p = stats.mannwhitneyu(m1, m0).pvalue if len(m1) >= 3 and len(m0) >= 3 else np.nan
    sub_rows.append({"miRNA": mir, "stage_kruskal_p": kw_p, "M1_vs_M0_p": mw_p,
                     "M1_minus_M0_log2": m1.mean() - m0.mean() if len(m1) else np.nan})
sub_res = pd.DataFrame(sub_rows).set_index("miRNA")
sub_res["stage_FDR"] = bh(sub_res["stage_kruskal_p"])
sub_res = sub_res.sort_values("stage_kruskal_p")
sub_res.to_csv("results/tissue_stage_metastasis.csv")
print("Metastasis groups:", sub["metastasis"].value_counts().to_dict(), "(M1 is small in TCGA-LIHC; interpret with care)")
sub_res.head(15)

In [ ]:
show = [m for m in ["hsa-mir-106b", "hsa-mir-124-1", "hsa-mir-124-2", "hsa-mir-124-3"] if m in log_expr.index]
show = (show + [m for m in sub_res.index if m not in show])[:4]
stages = ["I", "II", "III", "IV"]
if show:
    fig, axes = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3.2), squeeze=False)
    for ax, mir in zip(axes[0], show):
        vals = pd.Series(log_expr.loc[mir, case_to_sample.loc[tum_cases]].values, index=tum_cases)
        data = [vals[sub["stage_group"] == s].values for s in stages]
        ax.boxplot(data, showfliers=False)
        ax.set_xticks(range(1, len(stages) + 1)); ax.set_xticklabels([f"{s}\n(n={len(d)})" for s, d in zip(stages, data)])
        ax.set_title(mir, fontsize=9); ax.set_ylabel("log2(RPM + 1)")
    fig.suptitle("Tumour expression by AJCC stage", fontsize=10)
    fig.tight_layout(); fig.savefig("plots/tissue_stage_boxplots.png"); plt.show()

## 5. Circulating serum miRNAs (GSE113740)

In [ ]:
def geo_series_matrix(gse):
    stub = gse[:-3] + "nnn"
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{gse}/matrix/{gse}_series_matrix.txt.gz"
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    lines = gzip.decompress(r.content).decode("utf-8", errors="replace").splitlines()
    meta, table, in_table = {}, [], False
    for line in lines:
        if line.startswith("!series_matrix_table_begin"):
            in_table = True; continue
        if line.startswith("!series_matrix_table_end"):
            break
        if in_table:
            table.append(line)
        elif line.startswith("!Sample_"):
            key, *vals = line.split("\t")
            meta.setdefault(key, []).append([v.strip('"') for v in vals])
    expr = pd.read_csv(io.StringIO("\n".join(table)), sep="\t", index_col=0)
    expr.index = expr.index.astype(str).str.strip('"')
    expr.columns = expr.columns.str.strip('"')
    pheno = pd.DataFrame(index=expr.columns)
    for key in ["!Sample_title", "!Sample_source_name_ch1", "!Sample_platform_id"]:
        if key in meta:
            pheno[key.replace("!Sample_", "")] = meta[key][0]
    for row in meta.get("!Sample_characteristics_ch1", []):
        for col, v in zip(pheno.index, row):
            if ": " in v:
                k, val = v.split(": ", 1)
                pheno.loc[col, k.strip().lower()] = val.strip()
    return expr, pheno


def geo_platform_names(gpl):
    url = f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gpl}&targ=self&form=text&view=data"
    text = requests.get(url, timeout=300).text
    body = text.split("!platform_table_begin")[1].split("!platform_table_end")[0].strip()
    tab = pd.read_csv(io.StringIO(body), sep="\t", dtype=str)
    name_col = max(tab.columns, key=lambda c: tab[c].astype(str).str.startswith("hsa-").mean())
    return tab.set_index(tab.columns[0])[name_col]


serum, pheno = geo_series_matrix(GEO_ID)
serum = serum.apply(pd.to_numeric, errors="coerce")
if not serum.index.str.startswith("hsa-").mean() > 0.5:
    names = geo_platform_names(pheno["platform_id"].iloc[0])
    serum = serum[serum.index.isin(names.index)]
    serum.index = names.loc[serum.index].values
serum = serum[serum.index.str.startswith("hsa-")]
serum = serum.groupby(level=0).mean()
if np.nanmax(serum.values) > 50:          # raw intensities -> log2
    serum = np.log2(serum.clip(lower=1))
print(f"Serum matrix: {serum.shape[0]} miRNAs x {serum.shape[1]} samples")
print("Sample annotation fields:", [c for c in pheno.columns])

**Check the group column below.** The notebook picks the annotation field that names the disease group. If the automatic choice looks wrong, set `GROUP_FIELD` by hand.

In [ ]:
GROUP_FIELD = None   # e.g. "disease state"; None = choose automatically
if GROUP_FIELD is None:
    cand = [c for c in pheno.columns if re.search(r"disease|diagnos|group|status|tissue", c, re.I)]
    cand += ["source_name_ch1", "title"]
    GROUP_FIELD = next((c for c in cand if c in pheno and pheno[c].nunique() < 30), "source_name_ch1")
print("Group field:", GROUP_FIELD)
print(pheno[GROUP_FIELD].value_counts())

grp = pheno[GROUP_FIELD].astype(str)
is_hcc = grp.str.contains(r"hepatocellular|\bHCC\b", case=False, regex=True)
is_non_cancer = grp.str.contains(r"non[- ]?cancer|healthy|normal|control|benign", case=False)
is_other_cancer = (grp.str.contains(r"cancer|carcinoma|tumou?r|sarcoma|lymphoma|leuk|myeloma", case=False)
                   & ~is_hcc & ~is_non_cancer)
is_control = ~is_hcc & ~is_other_cancer
print(f"HCC: {is_hcc.sum()}  Non-cancer controls: {is_control.sum()}  Other cancers (excluded): {is_other_cancer.sum()}")

serum_de = welch_de(serum, pheno.index[is_hcc], pheno.index[is_control]).rename(
    columns={"mean_A": "mean_HCC", "mean_B": "mean_control"})
serum_sig = serum_de[(serum_de["FDR"] < FDR_CUTOFF) & (serum_de["log2FC"].abs() >= LOG2FC_CUTOFF)]
serum_de.to_csv("results/serum_DE_HCC_vs_control.csv")
print(f"{len(serum_sig)} serum miRNAs differ between HCC and non-cancer controls")
print("\nmiRNAs highlighted in the paper:")
serum_de.reindex(PAPER_SERUM_MIRNAS)

In [ ]:
show = [m for m in PAPER_SERUM_MIRNAS if m in serum.index] or list(serum_sig.index[:3])
if show:
    labels = ["Control", "HCC"]
    fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 3.2), squeeze=False)
    for ax, mir in zip(axes[0], show):
        data = [serum.loc[mir, pheno.index[is_control]].dropna(), serum.loc[mir, pheno.index[is_hcc]].dropna()]
        ax.boxplot(data, showfliers=False)
        ax.set_xticks([1, 2]); ax.set_xticklabels([f"{l}\n(n={len(d)})" for l, d in zip(labels, data)])
        ax.set_title(f"{mir}\nFDR = {serum_de.loc[mir, 'FDR']:.2g}", fontsize=9); ax.set_ylabel("log2 signal")
    fig.tight_layout(); fig.savefig("plots/serum_boxplots.png"); plt.show()

### Serum miRNAs vs AFP and stage (HCC patients)
Runs only if GSE113740's sample annotation includes AFP or stage fields.

In [ ]:
afp_col = next((c for c in pheno.columns if re.search(r"afp|fetoprotein", c, re.I)), None)
stage_col = next((c for c in pheno.columns if re.search(r"stage", c, re.I)), None)
hcc_ids = pheno.index[is_hcc]
cands = list(dict.fromkeys([m for m in PAPER_SERUM_MIRNAS if m in serum.index] + list(serum_sig.index[:20])))
clin_rows = []
for mir in cands:
    row = {"miRNA": mir}
    x = serum.loc[mir, hcc_ids]
    if afp_col:
        afp = pd.to_numeric(pheno.loc[hcc_ids, afp_col].str.extract(r"([\d.]+)", expand=False), errors="coerce")
        ok = x.notna() & afp.notna()
        if ok.sum() >= 10:
            row["AFP_spearman_rho"], row["AFP_p"] = stats.spearmanr(x[ok], afp[ok])
    if stage_col:
        st = pheno.loc[hcc_ids, stage_col]
        groups = [x[st == s].dropna().values for s in st.dropna().unique() if (st == s).sum() >= 3]
        if len(groups) >= 2:
            row["stage_kruskal_p"] = stats.kruskal(*groups).pvalue
    clin_rows.append(row)
serum_clin = pd.DataFrame(clin_rows).set_index("miRNA")
if serum_clin.shape[1]:
    serum_clin.to_csv("results/serum_AFP_stage.csv")
    print(f"AFP field: {afp_col}   Stage field: {stage_col}")
    display(serum_clin)
else:
    print("No AFP or stage fields in the GEO annotation; this step was skipped.")

## 6. Summary

In [ ]:
summary = pd.Series({
    "TCGA tumour samples": len(tumour),
    "TCGA normal samples": len(normal),
    "miRNAs analysed (tissue)": log_expr.shape[0],
    "Significant tissue miRNAs": len(sig),
    "Survival-associated (Cox p < 0.05)": int((surv_res["cox_p"] < 0.05).sum()),
    "Serum HCC samples": int(is_hcc.sum()),
    "Serum control samples": int(is_control.sum()),
    "Significant serum miRNAs": len(serum_sig),
})
summary.to_csv("results/summary.csv", header=["value"])
summary